# vLLM

**¿Qué es vLLM?**

Una infraestructura de ejecución de LLMs que optimiza el KV-Cache de la arquitectura Transformer para minimizar el tiempo de inferencia de un LLM.


**¿Qué puedo hacer con vLLM?**

Correr todos los modelos disponibles en HuggingFace a una velocidad considerablemente mayor que la implementación base. vLLM hereda todos los parámetros clásicos de generación, simplifica el código, y permite mejor compatibilidad entre GPU y CPU.

**¿Qué necesito para usar vLLM?**

Cuenta en HuggingFace con token de acceso válido (Es necesario para poder acceder a repositorios privados, como Llama4, y algunos datasets). Para requerimientos técnicos favor de revisar: https://docs.vllm.ai/en/v0.5.0/getting_started/installation.html 


**Consideraciones**

Si bien vLLM agiliza mucho la inferencia de un modelo, la capacidad de descargar e implementar un modelo sigue dependiendo de la GPU con la que estés trabajando. En las GPUs que tenemos en el GIL podemos correr modelos de hasta 20B de parámetros (aunque depende de la arquitectura interna del modelo). Naturalmente, entre más pequeño el modelo más rápida será la inferencia optimizada. 

**Seguridad**

Para acceder a los modelos de HuggingFace es necesario tener un token de acceso válido. Este token se declara como parámetro de la clase ``LLM()``. **IMPORTANTE**: Si van a subir un notebook o archivo .py a un repositorio PÚBLICO, TIENEN QUE BORRAR EL TOKEN ANTES DE SUBIR EL ARCHIVO (antes de hacer git push). Por cuestiones de seguridad, Github no permite que se suban archivos con este tipo de tokens (tokens de HuggingFace, claves de API de OpenAI/Claude, etc) y va a negar el push. (No es el fin del mundo revertir el commit, pero te ahorras muchos problemas si borras el token antes). 

In [1]:
from vllm import LLM, SamplingParams

model_id = 'google/gemma-3-4b-it'
max_m_len = 6000

# Para más información: https://docs.vllm.ai/en/v0.6.4/dev/sampling_params.html
sampling_params = SamplingParams(temperature = 0.67, top_p = 0.9, max_tokens = 5000, seed = 420)

# Para más información: https://docs.vllm.ai/en/v0.7.2/api/offline_inference/llm.html
llm = LLM(
    model = model_id,
    max_model_len = max_m_len,
    quantization = None,
    enforce_eager = True,
    gpu_memory_utilization = 0.9,
    #hf_token = 'PONGAN SU TOKEN', #OBS: ESTO ES LO QUE DEBEN BORRAR A LA HORA DE SUBIR EL CÓDIGO.
    limit_mm_per_prompt = {"image": 0, "video": 0} # Este parámetro es importante ya que limita o libera a un modelo multi-modal. Si dejamos tanto image como video en 0, el modelo trabaja solo con texto.
)

INFO 04-15 11:50:09 [utils.py:233] non-default args: {'max_model_len': 6000, 'disable_log_stats': True, 'enforce_eager': True, 'limit_mm_per_prompt': {'image': 0, 'video': 0}, 'model': 'google/gemma-3-4b-it'}
INFO 04-15 11:51:30 [model.py:533] Resolved architecture: Gemma3ForConditionalGeneration
INFO 04-15 11:51:30 [model.py:1582] Using max model len 6000
INFO 04-15 11:51:30 [scheduler.py:231] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 04-15 11:51:30 [vllm.py:754] Asynchronous scheduling is enabled.
WARNING 04-15 11:51:30 [vllm.py:788] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
WARNING 04-15 11:51:30 [vllm.py:799] Inductor compilation was disabled by user settings, optimizations settings that are only active during inductor compilation will be ignored.
INFO 04-15 11:51:31 [vllm.py:964] Cudagraph is disabled under eager mode
WARNING 04-15 11:51:31 [cuda.py:183] Forcing --disable_c

Loading safetensors checkpoint shards:   0% Completed | 0/2 [00:00<?, ?it/s]


(EngineCore pid=2041186) INFO 04-15 11:53:03 [default_loader.py:384] Loading weights took 6.46 seconds
(EngineCore pid=2041186) INFO 04-15 11:53:04 [gpu_model_runner.py:4566] Model loading took 7.79 GiB memory and 87.471306 seconds
(EngineCore pid=2041186) INFO 04-15 11:53:06 [gpu_worker.py:456] Available KV cache memory: 19.54 GiB
(EngineCore pid=2041186) WARNING 04-15 11:53:06 [kv_cache_utils.py:1056] Add 1 padding layers, may waste at most 3.45% KV cache memory
(EngineCore pid=2041186) INFO 04-15 11:53:06 [kv_cache_utils.py:1316] GPU KV cache size: 146,368 tokens
(EngineCore pid=2041186) INFO 04-15 11:53:06 [kv_cache_utils.py:1321] Maximum concurrency for 6,000 tokens per request: 24.34x
(EngineCore pid=2041186) INFO 04-15 11:53:06 [core.py:281] init engine (profile, create kv cache, warmup model) took 1.94 seconds
(EngineCore pid=2041186) WARNING 04-15 11:53:06 [vllm.py:788] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.c

In [2]:
# Text-Only

prompts = [
    "¿Quién es Gerard Way?",
    "What is DeepSeek-R1?",
    "Do cats know they are funny critters?"
]

outputs = llm.generate(prompts, sampling_params)

for output in outputs:
    prompt = output.prompt
    gen_text = output.outputs[0].text
    print("Prompt: {}. Generated text: {}".format(prompt, gen_text))

Rendering prompts:   0%|          | 0/3 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/3 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Prompt: ¿Quién es Gerard Way?. Generated text: 

Gerard Way es un escritor, músico y actor británico-estadounidense. Es mejor conocido como el vocalista principal y escritor principal de la banda de rock alternativo My Chemical Romance. También es el autor de la novela gráfica "The True Lives of the Dead".

Aquí hay algunos datos adicionales sobre él:

*   **Nacimiento:** 21 de noviembre de 1981 (42 años)
*   **Lugar de nacimiento:** Welwyn Garden City, Hertfordshire, Inglaterra, Reino Unido
*   **Educación:** Universidad de Tisch, Nueva York University
*   **Carrera:**
    *   Miembro fundador y vocalista principal de My Chemical Romance (2001-2013, 2019-presente)
    *   Autor de "The True Lives of the Dead" (2012)
    *   Actor (participaciones en series de televisión y películas)
    *   Guionista y productor de la serie de televisión "Doom Patrol" (2019-2023)

Si quieres saber más, puedes buscar información en:

*   **Wikipedia:** [https://es.wikipedia.org/wiki/Gerard_Way](https:/

**¿Qué cambia con un modelo multimodal?**

Algunas partes del preprocesamiento de la imagen/video, va a ser necesario usar PIL ( https://github.com/python-pillow/Pillow / https://pillow.readthedocs.io/en/stable/ ) para importar y procesar las imágenes en cuestión. De igual forma vLLM tiene ciertas clases que permiten un manejo adecuado. El siguiente bloque de código tiene el funcionamiento.

In [ ]:
# Vision Language Models
# Referencia: https://docs.vllm.ai/en/v0.7.2/getting_started/examples/vision_language.html

import random
from vllm.assets.image import ImageAsset
from vllm.assets.video import VideoAsset

def run_qwen_vl(question: str, modality: str):
    assert modality == "image"

    llm = LLM(
        model="Qwen/Qwen-VL",
        trust_remote_code=True,
        max_model_len=1024,
        max_num_seqs=2,
        disable_mm_preprocessor_cache=args.disable_mm_preprocessor_cache,
    )

    prompt = f"{question}Picture 1: <img></img>\n"
    stop_token_ids = None
    return llm, prompt, stop_token_ids


model_example_map = {
    "qwen_vl": run_qwen_vl
}

def get_multi_modal_input(args):
    """
    return {
        "data": image or video,
        "question": question,
    }
    """
    if args.modality == "image":
        # Input image and question
        image = ImageAsset("cherry_blossom") \
            .pil_image.convert("RGB")
        img_question = "What is the content of this image?"

        return {
            "data": image,
            "question": img_question,
        }

    if args.modality == "video":
        # Input video and question
        video = VideoAsset(name="sample_demo_1.mp4",
                           num_frames=args.num_frames).np_ndarrays
        vid_question = "Why is this video funny?"

        return {
            "data": video,
            "question": vid_question,
        }

    msg = f"Modality {args.modality} is not supported."
    raise ValueError(msg)


def apply_image_repeat(image_repeat_prob, num_prompts, data, prompt, modality):
    """Repeats images with provided probability of "image_repeat_prob". 
    Used to simulate hit/miss for the MM preprocessor cache.
    """
    assert (image_repeat_prob <= 1.0 and image_repeat_prob >= 0)
    no_yes = [0, 1]
    probs = [1.0 - image_repeat_prob, image_repeat_prob]

    inputs = []
    cur_image = data
    for i in range(num_prompts):
        if image_repeat_prob is not None:
            res = random.choices(no_yes, probs)[0]
            if res == 0:
                # No repeat => Modify one pixel
                cur_image = cur_image.copy()
                new_val = (i // 256 // 256, i // 256, i % 256)
                cur_image.putpixel((0, 0), new_val)

        inputs.append({
            "prompt": prompt,
            "multi_modal_data": {
                modality: cur_image
            }
        })

    return inputs


def main(args):
    model = args.model_type
    if model not in model_example_map:
        raise ValueError(f"Model type {model} is not supported.")

    modality = args.modality
    mm_input = get_multi_modal_input(args)
    data = mm_input["data"]
    question = mm_input["question"]

    llm, prompt, stop_token_ids = model_example_map[model](question, modality)

    # We set temperature to 0.2 so that outputs can be different
    # even when all prompts are identical when running batch inference.
    sampling_params = SamplingParams(temperature=0.2,
                                     max_tokens=64,
                                     stop_token_ids=stop_token_ids)

    assert args.num_prompts > 0
    if args.num_prompts == 1:
        # Single inference
        inputs = {
            "prompt": prompt,
            "multi_modal_data": {
                modality: data
            },
        }

    else:
        # Batch inference
        if args.image_repeat_prob is not None:
            # Repeat images with specified probability of "image_repeat_prob"
            inputs = apply_image_repeat(args.image_repeat_prob,
                                        args.num_prompts, data, prompt,
                                        modality)
        else:
            # Use the same image for all prompts
            inputs = [{
                "prompt": prompt,
                "multi_modal_data": {
                    modality: data
                },
            } for _ in range(args.num_prompts)]

    if args.time_generate:
        import time
        start_time = time.time()
        outputs = llm.generate(inputs, sampling_params=sampling_params)
        elapsed_time = time.time() - start_time
        print("-- generate time = {}".format(elapsed_time))

    else:
        outputs = llm.generate(inputs, sampling_params=sampling_params)

    for o in outputs:
        generated_text = o.outputs[0].text
        print(generated_text)